# SwinJSCC practical inspection notebook

This is a small, teaching-focused walkthrough of **one image** through the Stage 1A communication stack.

It makes five things visible:

1. the floating-point latent tensor emitted by the trained SwinJSCC model;
2. how many raw image bits and theoretical QPSK channel uses a raw image would require;
3. tensor summaries for every leaf layer in the encoder and decoder;
4. the meaning of channel uses, SNR, and CBR for native JSCC, VQ, and JPEG;
5. how the K=256 codebook replaces latent vectors with 8-bit indices.

The notebook is an inspection tool, not a second implementation of the model. It imports your local repository and reads the exported Stage 1A artifacts.

## Mental model

```text
RGB image
   |
   +--> raw reference: pixels -> bits -> theoretical QPSK symbols
   |
   +--> SwinJSCC encoder -> floating-point latent -> real/imaginary symbols
   |                                      |
   |                                      +--> AWGN (SNR controls noise)
   |                                      |
   |                                      +--> SwinJSCC decoder -> reconstruction
   |
   +--> codebook: latent vectors -> nearest prototype index (0..255)
                                      |
                                      +--> packet bits -> LDPC/QPSK (digital path)
```

Important naming used below:

- `latent_rate` is the learned latent-channel budget (`32, 64, 96, 128, 192` in Stage 1A). It is **not** the LDPC code rate and it is not Mbps.
- `channel_uses` means complex wireless symbols used for one image representation.
- `CBR` means `channel_uses / (height * width * 3)`, a normalized resource ratio.
- Native SwinJSCC is continuous-symbol JSCC. VQ and JPEG use a digital LDPC+QPSK path in the Stage 1A CSV.

In [ ]:

from pathlib import Path
import csv
import inspect
import json
import math
import os
import sys
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch


# This is the only path you normally need to edit on your laptop.
PROJECT_DIR = Path(r'D:\Bunker\BaseCamp\WirelessCommunication\SemanticComms\SemanticSchedulerEnd2End')
STAGE1_OUTPUT = PROJECT_DIR / 'Notebooks' / 'Stage1A'
INSPECTION_OUTPUT = PROJECT_DIR / 'results' / 'Stage1A' / 'practical_inspection_local'

# The exported artifact contains the final SA+RA model. Set this if you have a separate Base export.
IMAGE_PATH_OVERRIDE = None

INSPECTION_OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
SNR_DB = 10.0
LATENT_RATE = 96
DISPLAY_VALUES = 12

print('Project directory:', PROJECT_DIR)
print('Stage 1 output:', STAGE1_OUTPUT)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
print('Device used for inspection:', DEVICE)



### 1. Use the local repository

This version does not clone anything and does not assume Kaggle. It reads the repository at `PROJECT_DIR` and imports source code from its `src/` directory.


In [ ]:

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'PROJECT_DIR does not exist: {PROJECT_DIR}. Edit the path in the first code cell.'
    )

src_directory = PROJECT_DIR / 'src'
external_directory = PROJECT_DIR / 'external' / 'SwinJSCC'
if not src_directory.exists():
    raise FileNotFoundError(f'Missing source directory: {src_directory}')
if not (external_directory / 'net').exists():
    raise FileNotFoundError(
        f'Missing external SwinJSCC source: {external_directory}. '
        'Run scripts/bootstrap_swinjscc.py from the repository first.'
    )

if str(src_directory) not in sys.path:
    sys.path.insert(0, str(src_directory))

print('Local source import path:', src_directory)
print('External SwinJSCC source:', external_directory)



### 2. Validate the local Stage 1A artifacts

The local run expects `stage1_manifest.json`, the exported SwinJSCC checkpoint files, the K=256 codebook, and optionally `stage1a_metrics.csv` under `Notebooks/Stage1A`.


In [ ]:

manifest_path = STAGE1_OUTPUT / 'stage1_manifest.json'
model_directory_candidates = [
    STAGE1_OUTPUT / 'models' / 'SwinJSCC',
    STAGE1_OUTPUT,
]
model_directory = next(
    (directory for directory in model_directory_candidates if (directory / 'stage1_swinjscc_full.pt').exists()),
    None,
)
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing manifest: {manifest_path}')
if model_directory is None:
    raise FileNotFoundError(
        f'Missing stage1_swinjscc_full.pt under {STAGE1_OUTPUT}. '
        'Check that the Stage 1A artifacts were copied to the laptop.'
    )

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
codebook_path = STAGE1_OUTPUT / 'models' / 'Codebooks' / 'swinjscc_tx_k256.joblib'
if not codebook_path.exists():
    codebook_path = STAGE1_OUTPUT / 'swinjscc_tx_k256.joblib'
metrics_path = STAGE1_OUTPUT / 'stage1a_metrics.csv'
if not metrics_path.exists():
    metrics_path = STAGE1_OUTPUT / 'metrics' / 'stage1a_metrics.csv'
if not metrics_path.exists():
    metrics_path = None

print('Model directory:', model_directory)
print('Codebook:', codebook_path if codebook_path.exists() else 'not found')
print('Metrics CSV:', metrics_path if metrics_path else 'not found')
print('Manifest architecture:')
pprint(manifest.get('architecture', manifest.get('model_config', {})))



### 3. Load one local image and the exported model

If the manifest rejects the laptop environment, that is a real compatibility warning: the checkpoint was exported with a different Python/PyTorch/runtime combination. Do not silently ignore it; record the mismatch before deciding whether local inference is acceptable.


In [ ]:
from EncDecPipeline.Models.SwinJSCC.model_loader import load_stage1_swinjscc
from EncDecPipeline.Models.SwinJSCC.stage1_runner import _image_artifact
from PIL import Image as PILImage
from IPython.display import display


def find_image() -> Path:
    if IMAGE_PATH_OVERRIDE is not None:
        return Path(IMAGE_PATH_OVERRIDE)

    image_roots = [
        Path('/kaggle/input'),
        PROJECT_DIR / 'Test' / 'images',
        PROJECT_DIR,
        Path.cwd(),
    ]
    extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
    for root in image_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if path.is_file() and path.suffix.lower() in extensions:
                return path
    raise FileNotFoundError('No image found. Set IMAGE_PATH_OVERRIDE to one image.')


loader_input = STAGE1_OUTPUT
adapter = load_stage1_swinjscc(loader_input, device=DEVICE)
IMAGE_PATH = find_image()
image_artifact, image_metadata = _image_artifact(IMAGE_PATH)

print('Image:', IMAGE_PATH)
print('Image tensor shape:', tuple(image_artifact.tensor.shape))
print('Image tensor dtype/device:', image_artifact.tensor.dtype, image_artifact.tensor.device)
display(PILImage.open(IMAGE_PATH).convert('RGB'))
print('Adapter configuration:')
pprint(vars(adapter.config) if hasattr(adapter.config, '__dict__') else adapter.config)
actual_variant = getattr(adapter.config, 'variant', 'unknown')
if 'SA' in str(actual_variant) or 'RA' in str(actual_variant):
    print('NOTE: this artifact is an SA+RA export. Supply a Base export to inspect Base-only weights.')
else:
    print('Model variant:', actual_variant)
print('Manifest architecture:')
pprint(manifest.get('architecture', manifest.get('model_config', {})))

## A. Inspect the input and the floating-point latent tensor

The encoder does not emit bits. It emits a learned tensor of real-valued features. The communication adapter later groups pairs of real values into one complex symbol. The first printout below is the actual tensor, not a rounded diagram.

In [ ]:
def tensor_stats(tensor: torch.Tensor, value_count: int = DISPLAY_VALUES) -> dict:
    detached = tensor.detach().float().cpu()
    flattened = detached.reshape(-1)
    values = flattened[:value_count].tolist()
    return {
        'shape': tuple(tensor.shape),
        'dtype': str(tensor.dtype),
        'device': str(tensor.device),
        'numel': int(tensor.numel()),
        'min': float(detached.min().item()),
        'max': float(detached.max().item()),
        'mean': float(detached.mean().item()),
        'std': float(detached.std(unbiased=False).item()),
        'first_values': values,
    }


print('Input tensor statistics:')
pprint(tensor_stats(image_artifact.tensor))

In [ ]:
# Encode once. At this point `latent` is before codebook quantization.
latent = adapter.encode(image_artifact, snr_db=SNR_DB, rate=LATENT_RATE)

print('Latent tensor statistics:')
pprint(tensor_stats(latent.tensor, value_count=DISPLAY_VALUES))
print('\nFirst latent slice:')
print(latent.tensor.detach().flatten()[:DISPLAY_VALUES].cpu())
print('\nLatent metadata:')
pprint(getattr(latent, 'metadata', {}))
print('\nLatent rate mask shape:', None if getattr(latent, 'rate_mask', None) is None else tuple(latent.rate_mask.shape))

## B. Trace every encoder and decoder leaf layer

Forward hooks let us observe tensors without modifying SwinJSCC. To keep the notebook readable, each layer reports shape, dtype, number of values, range, mean, standard deviation, and the first few values. This is the useful form of “every tensor”: it shows the complete tensor flow without dumping hundreds of megabytes into the notebook output.

In [ ]:
def get_encoder_decoder(model_adapter):
    if hasattr(model_adapter, '_require_built'):
        built = model_adapter._require_built()
        if isinstance(built, (tuple, list)) and len(built) >= 2:
            return built[0], built[1]

    encoder = getattr(model_adapter, 'encoder', None)
    decoder = getattr(model_adapter, 'decoder', None)
    if encoder is None or decoder is None:
        raise AttributeError('Could not locate encoder and decoder on the adapter.')
    return encoder, decoder


def first_tensor(value):
    if torch.is_tensor(value):
        return value
    if isinstance(value, (tuple, list)):
        for item in value:
            found = first_tensor(item)
            if found is not None:
                return found
    if isinstance(value, dict):
        for item in value.values():
            found = first_tensor(item)
            if found is not None:
                return found
    return None


def attach_leaf_hooks(module, records, prefix):
    handles = []
    for name, child in module.named_modules():
        if any(child.children()):
            continue
        layer_name = prefix + (name or '<root>')

        def capture(_module, _inputs, output, layer_name=layer_name):
            tensor = first_tensor(output)
            if tensor is None:
                records.append({'layer': layer_name, 'output': 'non-tensor'})
                return
            record = tensor_stats(tensor)
            record['layer'] = layer_name
            records.append(record)

        handles.append(child.register_forward_hook(capture))
    return handles


encoder, decoder = get_encoder_decoder(adapter)
encoder_records = []
decoder_records = []

encoder_hooks = attach_leaf_hooks(encoder, encoder_records, 'encoder.')
try:
    traced_latent = adapter.encode(image_artifact, snr_db=SNR_DB, rate=LATENT_RATE)
finally:
    for handle in encoder_hooks:
        handle.remove()


def decode_with_options(model_adapter, latent_artifact):
    parameters = inspect.signature(model_adapter.decode).parameters
    options = {}
    if 'snr_db' in parameters:
        options['snr_db'] = SNR_DB
    if 'rate' in parameters:
        options['rate'] = LATENT_RATE
    return model_adapter.decode(latent_artifact, **options)


decoder_hooks = attach_leaf_hooks(decoder, decoder_records, 'decoder.')
try:
    reconstruction = decode_with_options(adapter, traced_latent)
finally:
    for handle in decoder_hooks:
        handle.remove()

print('Captured encoder tensors:', len(encoder_records))
print('Captured decoder tensors:', len(decoder_records))
print('Reconstruction type:', type(reconstruction).__name__)

In [ ]:
def image_tensor_for_display(value):
    tensor = getattr(value, 'tensor', value)
    if not torch.is_tensor(tensor):
        return None
    tensor = tensor.detach().float().cpu()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        return None
    if tensor.shape[0] in (1, 3):
        tensor = tensor.permute(1, 2, 0)
    tensor = tensor.clamp(-1.0, 1.0)
    if tensor.min() < 0:
        tensor = (tensor + 1.0) / 2.0
    return tensor.numpy().clip(0.0, 1.0)


native_reconstruction_image = image_tensor_for_display(reconstruction)
if native_reconstruction_image is not None:
    figure, axis = plt.subplots(figsize=(6, 5))
    axis.imshow(native_reconstruction_image)
    axis.set_title('Native SwinJSCC reconstruction')
    axis.axis('off')
    plt.show()
else:
    print('Reconstruction did not expose a displayable tensor.')

In [ ]:
def print_layer_records(records, title):
    print('\n' + title)
    print('-' * len(title))
    for index, record in enumerate(records):
        print(f'[{index:03d}] {record}')


print_layer_records(encoder_records, 'Encoder leaf-layer tensors')

In [ ]:
print_layer_records(decoder_records, 'Decoder leaf-layer tensors')

## C. See what “channel consumption” means

There are two different reference calculations here:

1. **Raw RGB reference:** the uncompressed image has 8 bits per colour sample. If those bits were sent with uncoded QPSK, two bits would occupy one complex symbol. This is a transparent theoretical reference; Stage 1A did not run raw RGB through the Sionna baseline.
2. **Native SwinJSCC:** the learned real-valued latent is paired into complex symbols. The number of symbols is the native JSCC channel consumption for this image and latent rate.

In [ ]:
def broadcast_rate_mask(rate_mask, target_tensor):
    if rate_mask is None:
        return None
    mask = rate_mask.to(device=target_tensor.device)
    if mask.shape == target_tensor.shape:
        return mask.bool()
    if mask.ndim <= target_tensor.ndim and mask.shape[0] == target_tensor.shape[0] and mask.shape[-1] == target_tensor.shape[-1]:
        reshaped = mask.reshape(
            mask.shape[0], *([1] * (target_tensor.ndim - 2)), mask.shape[-1]
        )
        return reshaped.expand_as(target_tensor).bool()
    try:
        return mask.expand_as(target_tensor).bool()
    except RuntimeError:
        return None


input_tensor = image_artifact.tensor
batch_size, _, height, width = input_tensor.shape
raw_rgb_samples = int(height * width * 3)
raw_bits = raw_rgb_samples * 8
raw_uncoded_qpsk_uses = math.ceil(raw_bits / 2)

latent_tensor = traced_latent.tensor
rate_mask = broadcast_rate_mask(getattr(traced_latent, 'rate_mask', None), latent_tensor)
active_real_values = int(rate_mask.sum().item()) if rate_mask is not None else int(latent_tensor.numel())
native_channel_uses = math.ceil(active_real_values / 2)

print('Raw image samples:', raw_rgb_samples)
print('Raw RGB bits:', raw_bits)
print('Raw uncoded QPSK reference uses:', raw_uncoded_qpsk_uses)
print('Latent real values before pairing:', active_real_values)
print('Native SwinJSCC complex channel uses:', native_channel_uses)
print('Native CBR:', native_channel_uses / raw_rgb_samples)


def real_to_complex_per_image(real_tensor: torch.Tensor) -> torch.Tensor:
    flattened = real_tensor.reshape(real_tensor.shape[0], -1)
    if flattened.shape[1] % 2:
        flattened = flattened[:, :-1]
    split = flattened.shape[1] // 2
    return torch.complex(flattened[:, :split], flattened[:, split:])


native_symbols = real_to_complex_per_image(latent_tensor)
print('Complex symbol tensor shape:', tuple(native_symbols.shape))
print('First complex symbols:', native_symbols.flatten()[:DISPLAY_VALUES].detach().cpu())

In [ ]:
def add_awgn(symbols: torch.Tensor, snr_db: float) -> torch.Tensor:
    signal_power = symbols.abs().square().mean(dim=1, keepdim=True).clamp_min(1e-12)
    snr_linear = 10.0 ** (snr_db / 10.0)
    noise_power = signal_power / snr_linear
    noise_scale = torch.sqrt(noise_power / 2.0)
    real_noise = torch.randn_like(symbols.real) * noise_scale
    imag_noise = torch.randn_like(symbols.real) * noise_scale
    return symbols + torch.complex(real_noise, imag_noise)


noisy_symbols = add_awgn(native_symbols, SNR_DB)
print('AWGN demonstration SNR (dB):', SNR_DB)
print('Symbols before noise:', native_symbols.flatten()[:DISPLAY_VALUES].detach().cpu())
print('Symbols after noise:', noisy_symbols.flatten()[:DISPLAY_VALUES].detach().cpu())
print('Channel uses do not change when SNR changes; SNR changes the noise level.')

## D. Inspect the K=256 codebook

Vector quantization changes a continuous latent into discrete indices:

```text
latent vector -> nearest one of 256 learned prototypes -> integer index 0..255
```

Because `256 = 2**8`, each index needs 8 payload bits before packet headers, padding, CRC, LDPC, and modulation overhead. The codebook itself is shared model state; it is not retransmitted for every image.

In [ ]:
if codebook_path is None:
    print('Codebook file not found; skip this section or set STAGE1_OUTPUT_OVERRIDE.')
    vq = None
else:
    from EncDecPipeline.Codebooks.minibatch_kmeans_vq import MiniBatchKMeansVQ

    vq = MiniBatchKMeansVQ.load(codebook_path)
    codebook_packet = vq.encode(traced_latent)
    quantized_latent = vq.decode(codebook_packet, traced_latent)

    transmitted_indices = np.asarray(codebook_packet.transmitted)
    metadata = getattr(codebook_packet, 'metadata', {})

    def first_metadata_int(values, key):
        value = values.get(key, 0)
        if isinstance(value, (list, tuple, np.ndarray)):
            flattened = np.asarray(value).reshape(-1)
            return int(flattened[0]) if flattened.size else 0
        return int(value)

    packet_bits_per_image = first_metadata_int(metadata, 'packet_bits_per_image')
    payload_bits_per_image = first_metadata_int(metadata, 'payload_bits_per_image')
    latent_float_bits = int(traced_latent.tensor.numel() * traced_latent.tensor.element_size() * 8)
    quantization_mse = float(torch.mean((traced_latent.tensor - quantized_latent.tensor) ** 2).item())

    centers = getattr(getattr(vq, 'model', None), 'cluster_centers_', None)
    center_shape = None if centers is None else tuple(np.asarray(centers).shape)
    center_bits = None if centers is None else int(np.asarray(centers).size * np.asarray(centers).dtype.itemsize * 8)

    print('Codebook path:', codebook_path)
    print('Codebook center shape:', center_shape)
    print('Codebook storage bits (shared once):', center_bits)
    print('Index tensor shape:', transmitted_indices.shape)
    print('Unique indices used by this image:', int(np.unique(transmitted_indices).size))
    print('First indices:', transmitted_indices.flatten()[:DISPLAY_VALUES])
    print('Payload bits per image:', payload_bits_per_image)
    print('Packet bits per image (including packet fields):', packet_bits_per_image)
    print('Theoretical uncoded QPSK uses for packet:', math.ceil(packet_bits_per_image / 2))
    print('Latent floating-point bits:', latent_float_bits)
    print('Quantization MSE:', quantization_mse)
    if packet_bits_per_image:
        print('Float-latent bits / packet bits:', latent_float_bits / packet_bits_per_image)
    print('Packet metadata:')
    pprint(metadata)

    quantized_reconstruction = decode_with_options(adapter, quantized_latent)
    quantized_reconstruction_image = image_tensor_for_display(quantized_reconstruction)
    if quantized_reconstruction_image is not None:
        figure, axis = plt.subplots(figsize=(6, 5))
        axis.imshow(quantized_reconstruction_image)
        axis.set_title('Reconstruction after K=256 quantization')
        axis.axis('off')
        plt.show()

## E. Compare all Stage 1A transmissions

The CSV contains trial rows over methods, SNR values, rates, and JPEG qualities. The table below selects one SNR (`10 dB`) so that the comparison is readable. Digital rows include LDPC/QPSK framing; native rows are continuous-symbol JSCC rows. Keep those method families conceptually separate when interpreting the plot.

In [ ]:
def read_csv_rows(path: Path):
    with path.open('r', newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


def as_float(row, key):
    value = row.get(key, '')
    if value in ('', 'None', 'null', 'nan'):
        return None
    return float(value)


if metrics_path is None:
    print('Metrics CSV not found; skip comparison section.')
    metric_rows = []
else:
    metric_rows = read_csv_rows(metrics_path)
    selected_rows = [
        row for row in metric_rows
        if as_float(row, 'snr_db') is not None and abs(as_float(row, 'snr_db') - SNR_DB) < 1e-9
    ]

    method_names = sorted({row['method'] for row in selected_rows})
    summary_rows = []
    for method in method_names:
        method_rows = [row for row in selected_rows if row['method'] == method]
        successful_rows = [row for row in method_rows if row.get('frame_success', '').lower() == 'true']
        uses = [as_float(row, 'channel_uses') for row in successful_rows]
        psnr = [as_float(row, 'psnr') for row in successful_rows]
        uses = [value for value in uses if value is not None]
        psnr = [value for value in psnr if value is not None]
        summary_rows.append({
            'method': method,
            'trials': len(method_rows),
            'successful_trials': len(successful_rows),
            'success_rate': len(successful_rows) / len(method_rows) if method_rows else 0.0,
            'mean_channel_uses_successful': sum(uses) / len(uses) if uses else None,
            'mean_psnr_successful': sum(psnr) / len(psnr) if psnr else None,
        })

    print(f'Rows in CSV: {len(metric_rows)}')
    print(f'Rows at SNR={SNR_DB:g} dB: {len(selected_rows)}')
    print('Summary at selected SNR:')
    for row in summary_rows:
        print(row)

    figure, axis = plt.subplots(figsize=(10, 6))
    for method in method_names:
        method_rows = [row for row in selected_rows if row['method'] == method]
        x_values = [as_float(row, 'channel_uses') for row in method_rows if as_float(row, 'psnr') is not None]
        y_values = [as_float(row, 'psnr') for row in method_rows if as_float(row, 'psnr') is not None]
        if x_values and y_values:
            axis.scatter(x_values, y_values, s=28, alpha=0.7, label=method)
    axis.set_title(f'Stage 1A transmission trials at SNR={SNR_DB:g} dB')
    axis.set_xlabel('Complex channel uses per image')
    axis.set_ylabel('PSNR (dB)')
    axis.grid(True, alpha=0.25)
    axis.legend(fontsize=8)
    figure.tight_layout()
    comparison_plot_path = INSPECTION_OUTPUT / 'stage1a_channel_uses_vs_psnr.png'
    figure.savefig(comparison_plot_path, dpi=160)
    plt.show()
    print('Saved plot:', comparison_plot_path)

## F. Save the inspection trace

These files make the walkthrough auditable without rerunning the model:

- `latent_summary.json` — input/latent shapes and values;
- `encoder_layers.csv` and `decoder_layers.csv` — one row per traced leaf layer;
- `transmission_summary.json` — raw, native, and codebook accounting;
- `stage1a_channel_uses_vs_psnr.png` — comparison plot when the CSV is available.

In [ ]:
def write_layer_csv(path: Path, records):
    if not records:
        return
    fieldnames = sorted({key for record in records for key in record.keys()})
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


latent_summary = {
    'image_path': str(IMAGE_PATH),
    'input': tensor_stats(image_artifact.tensor),
    'latent': tensor_stats(traced_latent.tensor),
    'latent_metadata': getattr(traced_latent, 'metadata', {}),
    'latent_rate': LATENT_RATE,
    'snr_db': SNR_DB,
}
transmission_summary = {
    'raw_rgb_samples': raw_rgb_samples,
    'raw_bits': raw_bits,
    'raw_uncoded_qpsk_uses': raw_uncoded_qpsk_uses,
    'native_real_values': active_real_values,
    'native_complex_channel_uses': native_channel_uses,
    'native_cbr': native_channel_uses / raw_rgb_samples,
}
if vq is not None:
    transmission_summary.update({
        'codebook_path': str(CODEBOOK_PATH),
        'codebook_packet_bits_per_image': packet_bits_per_image,
        'codebook_payload_bits_per_image': payload_bits_per_image,
        'codebook_theoretical_uncoded_qpsk_uses': math.ceil(packet_bits_per_image / 2),
        'codebook_quantization_mse': quantization_mse,
    })

(INSPECTION_OUTPUT / 'latent_summary.json').write_text(
    json.dumps(latent_summary, indent=2, default=str), encoding='utf-8'
)
write_layer_csv(INSPECTION_OUTPUT / 'encoder_layers.csv', encoder_records)
write_layer_csv(INSPECTION_OUTPUT / 'decoder_layers.csv', decoder_records)
(INSPECTION_OUTPUT / 'transmission_summary.json').write_text(
    json.dumps(transmission_summary, indent=2, default=str), encoding='utf-8'
)
print('Inspection files written to:', INSPECTION_OUTPUT)

## Checks and interpretation

The most important sanity checks are:

1. the latent has floating-point values and a batch dimension;
2. native channel uses equal approximately half the active real latent values because two real values form one complex symbol;
3. raw QPSK uses are much larger because raw RGB has 24 bits per pixel;
4. codebook indices are integers in `0..255`, while quantized latent values remain vectors reconstructed from the shared prototypes;
5. a lower `latent_rate` generally reduces native resource use, but reconstruction quality must be read from PSNR/SSIM rather than assumed;
6. digital `frame_success=False` means the received packet failed its error-control check. Native JSCC does not have that packet-success column in the same sense.

This is a corrected, modern PyTorch port and an instrumentation notebook. It does not claim paper-exact replication or prove semantic task gains; Stage 1A is the image-communication backbone that later semantic parameters and scheduling will consume.